# Symptom normalization evaluation

Audits the real synonym/embedding normalization implementation. Challenge examples remain unscored until their expected canonical labels are manually reviewed.

## Vocabulary and stage configuration

In [2]:
from pathlib import Path
import sys
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'src').is_dir() and (p/'data').is_dir())
sys.path.insert(0,str(ROOT));sys.path.insert(0,str(ROOT/'results'))
from utils.evaluation import setup,save,meta
ROOT=setup(ROOT);OUT=ROOT/'results'
import pandas as pd
from src.disease_prediction.ner import ner_symptom_extraction as nse
vocab=nse.get_ordered_vocabulary();print('canonical symptoms',len(vocab),'aliases',len(nse.SYNONYM_MAP),'model',nse.EMBEDDING_MODEL_NAME,'similarity threshold',nse.SIMILARITY_THRESHOLD,'margin',nse.SIMILARITY_MARGIN)


canonical symptoms 131 aliases 123 model sentence-transformers/all-MiniLM-L6-v2 similarity threshold 0.7 margin 0.05


## Human-review challenge set

Edit `reviewed_expected` only after review. Proposed rows are not ground truth. Exact and dictionary matches can run offline; semantic matching may return UNMATCHED when its embedding model is unavailable.

In [4]:
cases=[('skin rashes','reordered'),('rashes on my skin','paraphrase'),('head ache','spelling'),('my tummy hurts','informal'),('shortness of breath','multiword'),('xyz symptom','unknown'),('pain','ambiguous'),('no headache','negation'),('severe joint pain','modifier')]
d=pd.DataFrame([{'mention':x,'category':k,'reviewed_expected':''} for x,k in cases]);d['prediction']=d.mention.map(nse.normalize_symptom);display(d);d.to_csv(OUT/'tables/normalization_challenge_review.csv',index=False); reviewed=d[d.reviewed_expected!=''];print('Scored n=',len(reviewed),'accuracy=',(reviewed.prediction==reviewed.reviewed_expected).mean() if len(reviewed) else 'NOT REPORTED: no reference labels')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,mention,category,reviewed_expected,prediction
0,skin rashes,reordered,,skin rash
1,rashes on my skin,paraphrase,,skin rash
2,head ache,spelling,,headache
3,my tummy hurts,informal,,UNMATCHED
4,shortness of breath,multiword,,breathlessness
5,xyz symptom,unknown,,UNMATCHED
6,pain,ambiguous,,UNMATCHED
7,no headache,negation,,headache
8,severe joint pain,modifier,,joint pain


Scored n= 0 accuracy= NOT REPORTED: no reference labels


## Coverage on annotated mentions

NER mention annotations have no canonical normalization labels, so only match coverage is measured, not accuracy.

In [5]:
import json
ner=pd.read_csv(ROOT/'data/ner/splits/ner_test.csv'); ms=[]
for raw in ner.mentions:
 try: ms+=json.loads(raw)
 except Exception: pass
if ms:
 out=[nse.normalize_symptom(str(x)) for x in ms];m={'mentions':len(ms),'unique_mentions':len(set(ms)),'matched_rate':sum(x!='UNMATCHED' for x in out)/len(out),'unmatched_rate':sum(x=='UNMATCHED' for x in out)/len(out)};print(m);pd.DataFrame({'mention':ms,'output':out}).to_csv(OUT/'tables/normalization_ner_coverage.csv',index=False);save(OUT/'metrics/normalization_coverage.json',meta(ROOT,'normalization',summary=m))
else:print('No annotated mentions found')


{'mentions': 150, 'unique_mentions': 96, 'matched_rate': 1.0, 'unmatched_rate': 0.0}
